# LingBot-Map — proper 3D reconstructed video (flythrough MP4)

This creates the **official LingBot demo-style reconstructed video**:
a rendered path through the 3D reconstruction of **your** clip.

| Output | Meaning |
|--------|---------|
| `*.mp4` | Proper reconstructed flythrough video |
| `*.glb` | Interactive 3D of the same scene |

Uses official `demo_render/batch_demo.py`.

## Before starting
1. Runtime → Change runtime type → **T4 GPU** (or better)
2. Use a **short indoor** video (20–60 seconds)
3. Run cells in order


## 0. Check GPU


In [ ]:
import torch
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Enable GPU: Runtime → Change runtime type → T4 GPU")
print("gpu", torch.cuda.get_device_name(0))


## 1. Clone LingBot-Map + install ffmpeg


In [ ]:
import os
from pathlib import Path
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
%cd /content
if not Path("/content/lingbot-map").exists():
    !git clone --depth 1 https://github.com/Robbyant/lingbot-map.git
%cd /content/lingbot-map
print("cwd", os.getcwd())


## 2. Install LingBot + render deps (+ Kaolin)


In [ ]:
import os, sys, torch
!{sys.executable} -m pip -q install -e ".[vis]" huggingface_hub
!{sys.executable} -m pip -q install "numpy==1.26.4" open3d==0.19.0 pyyaml onnxruntime-gpu tqdm opencv-python-headless

ver = torch.__version__.split("+")[0]
print("torch", torch.__version__)
kaolin_ok = False
for tag in ["cu128", "cu126", "cu124", "cu121"]:
    url = f"https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-{ver}_{tag}.html"
    print("try", url)
    rc = os.system(f'{sys.executable} -m pip -q install --index-url https://pypi.org/simple kaolin -f "{url}"')
    if rc == 0:
        try:
            import kaolin
            print("Kaolin OK")
            kaolin_ok = True
            break
        except Exception as e:
            print("kaolin import fail", e)
if not kaolin_ok:
    print("WARNING: Kaolin missing — flythrough may fail on this Colab image.")


## 3. Build CUDA render extensions


In [ ]:
%cd /content/lingbot-map/demo_render/render_cuda_ext
!python setup.py build_ext --inplace
%cd /content/lingbot-map


## 4. Download LingBot weights


In [ ]:
from huggingface_hub import hf_hub_download
MODEL_PATH = hf_hub_download(
    repo_id="robbyant/lingbot-map",
    filename="lingbot-map.pt",
    local_dir="/content/weights",
)
print(MODEL_PATH)


## 5. Upload your indoor video


In [ ]:
from google.colab import files
from pathlib import Path
IN = Path("/content/my_video")
IN.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
assert uploaded, "Upload one video"
name = list(uploaded.keys())[0]
VIDEO = IN / name
VIDEO.write_bytes(uploaded[name])
print(VIDEO, round(VIDEO.stat().st_size / 1e6, 2), "MB")


## 6. Run official batch_demo.py → reconstructed MP4

This is the LingBot README pipeline for indoor flythrough rendering.


In [ ]:
import os
from pathlib import Path

OUT = Path("/content/lingbot_out")
OUT.mkdir(parents=True, exist_ok=True)
CONFIG = "/content/lingbot-map/demo_render/config/indoor.yaml"
os.chdir("/content/lingbot-map")

cmd = (
    "python demo_render/batch_demo.py"
    f" --video_path '{VIDEO}'"
    f" --output_folder '{OUT}'"
    f" --model_path '{MODEL_PATH}'"
    f" --config {CONFIG}"
    " --mode windowed --window_size 64"
    " --keyframe_interval 4 --overlap_keyframes 8"
    " --camera_vis default --save_predictions"
)
print(cmd)
code = os.system(cmd)
print("exit", code)
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(p, f"{p.stat().st_size/1e6:.2f} MB")


## 7. Download the reconstructed video (+ GLB if present)


In [ ]:
from google.colab import files
from pathlib import Path

OUT = Path("/content/lingbot_out")
mp4s = sorted(OUT.rglob("*.mp4"))
glbs = sorted(OUT.rglob("*.glb"))
print("mp4s", mp4s)
print("glbs", glbs)
if not mp4s:
    raise SystemExit(
        "No MP4 produced. Check Kaolin/render errors above. "
        "Fallback GLB-only: notebooks/colab_lingbot_video_to_glb.ipynb"
    )
files.download(str(mp4s[0]))
if glbs:
    files.download(str(glbs[0]))
print("Done — this MP4 is the LingBot-style reconstructed flythrough.")
